# Full dataset

In [1]:
from datasets import load_dataset
from collections import Counter

CACHE_DIR = r"D:\Hand_Gesture\data\raw\hf_cache_full"

dataset = load_dataset(
    "testdummyvt/hagRIDv2_512px",
    split="train",
    cache_dir=CACHE_DIR,
)

label_names = dataset.features["label"].names
counts = Counter(dataset["label"])

print("Label names:")
for i, name in enumerate(label_names):
    print(i, name)

print("\nClass counts:")
for idx, count in sorted(counts.items()):
    print(idx, label_names[idx], count)

c:\Users\user\anaconda3\envs\handgesture\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Label names:
0 call
1 dislike
2 fist
3 four
4 grabbing
5 grip
6 hand_heart
7 hand_heart2
8 holy
9 like
10 little_finger
11 middle_finger
12 mute
13 no_gesture
14 ok
15 one
16 palm
17 peace
18 peace_inverted
19 point
20 rock
21 stop
22 stop_inverted
23 take_picture
24 three
25 three2
26 three3
27 three_gun
28 thumb_index
29 thumb_index2
30 timeout
31 two_up
32 two_up_inverted
33 xsign

Class counts:
0 call 20061
1 dislike 23624
2 fist 23543
3 four 23436
4 grabbing 28352
5 grip 28406
6 hand_heart 21576
7 hand_heart2 23986
8 holy 31402
9 like 23244
10 little_finger 28301
11 middle_finger 30034
12 mute 24349
13 no_gesture 1464
14 ok 23153
15 one 23872
16 palm 23710
17 peace 23801
18 peace_inverted 21849
19 point 29679
20 rock 24182
21 stop 23268
22 stop_inverted 22300
23 take_picture 20767
24 three 22721
25 three2 21626
26 three3 32354
27 three_gun 29543
28 thumb_index 38995
29 thumb_index2 10916
30 timeout 21679
31 two_up 22688
32 two_up_inverted 21991
33 xsign 30586


In [2]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm
import random

label_names = dataset.features["label"].names
labels = dataset["label"]

target_map = {
    "fist": 1,
    "like": 2,
    "ok": 3,
    "one": 4,
    "palm": 5,
}

TARGET_PER_CLASS = 1000
NA_COUNT = 5000

out_dir = Path("D:/Hand_Gesture/data/processed_full_10k")
crop_dir = out_dir / "crops"
lm_dir = out_dir / "landmarks"

crop_dir.mkdir(parents=True, exist_ok=True)
lm_dir.mkdir(parents=True, exist_ok=True)

selected_indices = []

for name, new_label in target_map.items():
    original_label_id = label_names.index(name)
    idxs = [i for i, y in enumerate(labels) if y == original_label_id]
    selected_indices += [(idx, new_label, name) for idx in idxs[:TARGET_PER_CLASS]]

target_label_ids = {label_names.index(name) for name in target_map.keys()}
na_indices = [i for i, y in enumerate(labels) if y not in target_label_ids]

random.seed(0)
na_indices = random.sample(na_indices, NA_COUNT)

selected_indices += [(idx, 0, "N_A") for idx in na_indices]

print("total selected:", len(selected_indices))

total selected: 10000


In [5]:
from hand_preprocess import MediaPipeHandPreprocessor

preprocessor = MediaPipeHandPreprocessor()

In [6]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm

out_dir = Path("D:/Hand_Gesture/data/processed_full_10k")
crop_dir = out_dir / "crops"
lm_dir = out_dir / "landmarks"

crop_dir.mkdir(parents=True, exist_ok=True)
lm_dir.mkdir(parents=True, exist_ok=True)

records = []
fail_count = 0

for idx, new_label, new_label_name in tqdm(selected_indices):
    sample = dataset[idx]
    img = sample["image"]
    original_label = sample["label"]
    original_label_name = label_names[original_label]

    result = preprocessor.preprocess_image(img)

    if result is None:
        fail_count += 1
        continue

    crop, landmarks = result

    base_name = f"{idx}_{new_label_name}"

    crop_path = crop_dir / f"{base_name}.png"
    lm_path = lm_dir / f"{base_name}.npy"

    Image.fromarray(crop).save(crop_path)
    np.save(lm_path, landmarks)

    records.append({
        "idx": idx,
        "original_label": original_label,
        "original_label_name": original_label_name,
        "label": new_label,
        "label_name": new_label_name,
        "crop_path": str(crop_path),
        "landmark_path": str(lm_path),
        "quality": "ok"
    })

df = pd.DataFrame(records)
df.to_csv(out_dir / "labels.csv", index=False)

print("selected:", len(selected_indices))
print("saved:", len(df))
print("failed no hand:", fail_count)
print("labels.csv:", out_dir / "labels.csv")

100%|██████████| 10000/10000 [03:52<00:00, 42.95it/s]

selected: 10000
saved: 9207
failed no hand: 793
labels.csv: D:\Hand_Gesture\data\processed_full_10k\labels.csv
